# Fine-tune DistilBERT on Vertex AI Workbench (T4)

Run this notebook **on the gated Workbench instance**, not locally and
not on a Dataflow worker.

1. `n1-standard-8` + `NVIDIA_TESLA_T4` × 1 in `europe-central2-b`
2. Dataset cache: `gs://co-tf-artifacts-dev/nlp/datasets/`
3. MLflow: `gs://co-tf-artifacts-dev/nlp/mlruns`
4. **Stop the instance when the run finishes** — idle shutdown is 3 hours,
   GPU billing is not free in the meantime.

Model: `distilbert-base-uncased`, 3-class (`neg`/`neu`/`pos`), `MAX_LEN=128`.

In [ ]:
import os
from pathlib import Path

# Repo root on the Workbench VM. Adjust if you cloned elsewhere.
REPO = Path.home() / "pop-vibe-check"
os.chdir(REPO)
%pip install -q -r nlp/training/requirements.txt

## Cache datasets from GCS

Hugging Face `cache_dir` is local. Rsync the shared prefix so a restarted
Workbench does not hit the Hub again. First session: download via the
loaders, then `gsutil -m rsync -r /home/jupyter/hf-datasets gs://co-tf-artifacts-dev/nlp/datasets/`.

In [ ]:
from nlp.training.loaders import DEFAULT_GCS_DATASETS_URI, cache_dir_from_gcs

CACHE = cache_dir_from_gcs(DEFAULT_GCS_DATASETS_URI, Path.home() / "hf-datasets")
print("cache", CACHE)

## Train

`train.py` loads SST-2 + Twitter (~100k) + GoEmotions (Reddit substitute)
+ optional `--own-domain` gold JSONL, fine-tunes DistilBERT, logs to MLflow.

In [ ]:
from nlp.training.train import main

main([
    "--cache-dir", str(CACHE),
    "--output-dir", str(Path.home() / "models" / "distilbert-sent"),
    "--epochs", "3",
    "--batch-size", "16",
    "--lr", "2e-5",
])

## After training

1. Copy the export to GCS:
   `gsutil -m cp -r ~/models/distilbert-sent gs://co-tf-artifacts-dev/nlp/models/`
2. `python -m nlp.endpoint.register upload --model-dir gs://…`
3. Enable the Endpoint in Terraform, then `register deploy`.
4. **Stop this Workbench instance** (`gcloud workbench instances stop …` or
   `desired_state=STOPPED`). Do not leave the T4 running overnight.

See `nlp/README.md` and `terraform/modules/vertex_nlp/README.md`.